# AI Engine 
### Les 4 Agents IA du tuteur Study Buddy (méthode socratique)

| Agent | Fonction | Rôle |
|---|---|---|
| Diagnostiqueur | `agent_diagnostiqueur(question)` | Évalue le niveau de l'étudiant de départ|
| Tuteur Socratique | `agent_tuteur_socratique(question, chunks, historique)` | (Coeur du projet) Guide SANS donner la réponse, l'étudiant apprend |
| Évaluateur | `agent_evaluateur(reponse_etudiant, question)` | Analyse la compréhension, est-ce que l'étudiant a progressé grace agent 2 |
| Générateur d'indices | `generer_indice(question, niveau_indice)` | Donne un coup de pouce quand l'étudiant est bloqué dans agent 2 (3 niveaux d'aide : vague puis précis puis donne presque la réponse) |


## Installation des dépendances pour l'API

pip install groq -q
pip show groq

## Imports

In [3]:
import os
from typing import List, Dict
from groq import Groq

## Configuration de l'API Groq 

In [4]:
import os
from dotenv import load_dotenv
from groq import Groq

# https://console.groq.com/keys
load_dotenv()  # lit le .env

# Modèle global
MODEL = "llama-3.3-70b-versatile"

# Client Groq (chargé une seule fois) 
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print(f"Client Groq initialisé avec le modèle : {MODEL}")

# LLM réutilisé par les 4 agents
def _appeler_llm(system_prompt, user_prompt, temperature=0.7):
    reponse = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return reponse.choices[0].message.content

Client Groq initialisé avec le modèle : llama-3.3-70b-versatile


## Agent 1 : `agent_diagnostiqueur(question)`
Évalue le niveau de l'étudiant en posant des questions de diagnostic.

**Rôle :** Avant de commencer la session, cet agent pose 2-3 questions courtes pour savoir si l'étudiant est **débutant**, **intermédiaire** ou **avancé** sur le sujet.

In [6]:
# 1. Poser des questions a l'etudiant
def agent_diagnostiqueur(question: str) -> str:
    """
    Agent Diagnostiqueur — évalue le niveau de l'étudiant.

    Pose 1 question courte pour savoir si l'étudiant
    est débutant, intermédiaire ou avancé sur le sujet demandé.

    Paramètres
    ----------
    question : str — la question ou le sujet soumis par l'étudiant

    Retourne
    --------
    str — 1 question de diagnostic à poser à l'étudiant
    """
    system_prompt = """Tu es un agent diagnostiqueur pédagogique.
Ton rôle est d'évaluer le niveau de l'étudiant AVANT de commencer à lui enseigner.

Règles STRICTES :
- Pose exactement 1 question COURTE sur le sujet demandé.
- La question doit permettre de distinguer un débutant d'un étudiant avancé.
- Ne donne AUCUNE explication ni réponse. Pose seulement la question.
- Commence directement par la question, sans introduction.
- Langue : français."""

    user_prompt = f"L'étudiant veut apprendre sur : {question}"

    return _appeler_llm(system_prompt, user_prompt, temperature=0.5)

# 2. Analyser la reponse de l'etudiant pour connaitre son niveau
def analyser_niveau(matiere, reponses_etudiant):
    system_prompt = f"""Tu es un agent diagnostiqueur pédagogique strict.
La matière évaluée est : {matiere}

RÈGLE PRINCIPALE : En cas de doute, classe TOUJOURS au niveau inférieur.

DÉBUTANT — si l'étudiant :
- Dit "je ne sais pas" ou "je ne connais pas les détails"
- Donne seulement une réponse très vague sans explication
- Montre qu'il ne connaît pas les concepts de base de {matiere}

INTERMÉDIAIRE — si l'étudiant :
- Connaît les concepts de base de {matiere} et les explique correctement
- Utilise des mots simples mais justes
- Manque de précision sur les détails avancés de {matiere}

AVANCÉ — si l'étudiant :
- Utilise le vocabulaire technique précis de {matiere}
- Explique les mécanismes détaillés avec précision
- Montre une vraie maîtrise de {matiere}

Réponds TOUJOURS dans ce format exact :
NIVEAU : [Débutant / Intermédiaire / Avancé]
RAISON : [Une phrase courte]"""

    user_prompt = f"""Matière : {matiere}
Réponses de l'étudiant : {reponses_etudiant}"""

    return _appeler_llm(system_prompt, user_prompt, temperature=0.1)

print("agent_diagnostiqueur() défini, faire maintenant le test.")

agent_diagnostiqueur() défini, faire maintenant le test.


### Test de `agent_diagnostiqueur()`

In [7]:
MA_QUESTION = "la photosynthèse"    # modifie le sujet ici

# 1. Poser les questions
diagnostic = agent_diagnostiqueur(MA_QUESTION)
print(f"Question de diagnostic pour : « {MA_QUESTION} »")
print("=" * 60)
print(diagnostic)

Question de diagnostic pour : « la photosynthèse »
Quel est le rôle de la lumière dans le processus de photosynthèse ?


In [8]:
# 2. Reponses de l'etudiant
reponses_simulees = "Je ne sais pas vraiment, j'ai jamais entendu parler de ça."
print(f"\nRéponses simulées de l'étudiant :\n{reponses_simulees}")
# Debutant : Je ne sais pas vraiment, j'ai jamais entendu parler de ça.
# Intermediatre : Je connais les bases mais pas les détails techniques.
# Avance : Je maîtrise bien le sujet, je connais les concepts précis et le vocabulaire technique.

# Analyser la reponse de l'etudiant pour connaitre son niveau
print("\nNIVEAU DÉTECTÉ :")
niveau = analyser_niveau(MA_QUESTION, reponses_simulees)
print(niveau)


Réponses simulées de l'étudiant :
Je ne sais pas vraiment, j'ai jamais entendu parler de ça.

NIVEAU DÉTECTÉ :
NIVEAU : Débutant
RAISON : L'étudiant exprime son ignorance totale sur la matière.


## Agent 2 : `agent_tuteur_socratique(question, chunks, historique)`
Guide l'étudiant par des questions — **sans jamais donner la réponse directement**.

**Rôle :** C'est le cœur du projet. Cet agent utilise les passages extraits du document (fournis par Membre 1) et l'historique de la conversation pour poser des questions qui amènent l'étudiant à trouver la réponse lui-même.

In [9]:
def agent_tuteur_socratique(
    question: str,
    chunks: List[str],
    historique: List[Dict[str, str]],
) -> str:
    """
    Agent Tuteur Socratique — guide sans donner la réponse directement.

    Paramètres
    ----------
    question    : str — la question posée par l'étudiant
    chunks      : List[str] — passages pertinents extraits du document
                  (résultat de rechercher() depuis rag_pipeline.py)
    historique  : List[Dict] — historique de la conversation
                  format : [{"role": "etudiant"|"tuteur", "message": "..."}]

    Retourne
    --------
    str — une réponse guidante (question socratique ou piste de réflexion)
    """
    # Construit le contexte documentaire
    contexte_docs = "\n\n".join(
        [f"[Extrait {i+1}] {chunk}" for i, chunk in enumerate(chunks)]
    ) if chunks else "Aucun document disponible."

    # Construit l'historique de conversation
    historique_texte = "\n".join(
        [f"{msg['role'].upper()} : {msg['message']}" for msg in historique]
    ) if historique else "Début de la conversation."

    system_prompt = """Tu es un tuteur pédagogique qui utilise la méthode socratique.

RÈGLE ABSOLUE : Tu ne donnes JAMAIS la réponse directement.

À la place, tu :
1. Poses des questions qui amènent l'étudiant à réfléchir.
2. Fais des analogies simples avec des exemples du quotidien.
3. Mets en valeur ce que l'étudiant a déjà bien compris.
4. Orientes doucement vers la bonne piste sans la révéler.

Tu t'appuies sur les extraits du document pour guider (sans les copier mot pour mot).
Si l'étudiant donne une mauvaise réponse, tu ne dis pas "c'est faux".
Tu demandes : "Qu'est-ce qui te fait penser ça ?"
Langue : français. Ton : bienveillant et encourageant."""

    user_prompt = f"""QUESTION DE L'ÉTUDIANT : {question}

EXTRAITS DU DOCUMENT (pour guider ta réponse) :
{contexte_docs}

HISTORIQUE DE LA CONVERSATION :
{historique_texte}

Génère une réponse socratique courte (3-5 phrases maximum)."""

    return _appeler_llm(system_prompt, user_prompt, temperature=0.7)


print("agent_tuteur_socratique() défini, faire maintenant le test.")

agent_tuteur_socratique() défini, faire maintenant le test.


### Test de `agent_tuteur_socratique()`

In [10]:
question_test = "Comment les plantes fabriquent-elles leur nourriture ?"

# Documents de référence (viendraient de rag_pipeline.ipynb)
chunks_test = [
    "La photosynthèse est le processus par lequel les plantes convertissent la lumière solaire en énergie chimique.",
    "Les chloroplastes contiennent de la chlorophylle, qui absorbe la lumière. La plante utilise CO2 et eau.",
    "L'équation de la photosynthèse : 6CO2 + 6H2O + lumière → C6H12O6 + 6O2",
]

# Historique vide au départ
historique = []

print(f"Sujet : « {question_test} »")
print("=" * 60)
print("Tape 'stop' pour terminer la conversation\n")

while True:
    # L'étudiant écrit sa réponse
    reponse_etudiant = input("ÉTUDIANT : ")
    
    if reponse_etudiant.lower() == "stop":
        break
    
    # Ajoute la réponse de l'étudiant à l'historique
    historique.append({"role": "etudiant", "message": reponse_etudiant})
    
    # Le tuteur répond
    reponse_tuteur = agent_tuteur_socratique(question_test, chunks_test, historique)
    
    # Ajoute la réponse du tuteur à l'historique
    historique.append({"role": "tuteur", "message": reponse_tuteur})
    
    print(f"\nTUTEUR : {reponse_tuteur}\n")

# Affiche l'historique complet à la fin
print("\nHISTORIQUE COMPLET :")
print("=" * 60)
for msg in historique:
    if msg["role"] == "etudiant":
        print(f"ÉTUDIANT : {msg['message']}")
    else:
        print(f"TUTEUR   : {msg['message']}")

Sujet : « Comment les plantes fabriquent-elles leur nourriture ? »
Tape 'stop' pour terminer la conversation


HISTORIQUE COMPLET :


## Agent 3 : `agent_evaluateur(reponse_etudiant, question)`
Analyse si l'étudiant a compris apprend son apprentissage grace à agent 2, et détecte les points de confusion.

**Rôle :** Après que l'étudiant a répondu, cet agent évalue la qualité de la réponse et retourne un bilan structuré (niveau de compréhension + points à améliorer).

In [11]:
def agent_evaluateur(reponse_etudiant, question_initiale):
    system_prompt = """Tu es un évaluateur pédagogique strict.

RÈGLE PRINCIPALE : Évalue UNIQUEMENT ce que l'étudiant a écrit, pas ce qu'il aurait pu ajouter.

AVANCÉ — si l'étudiant :
- Cite les deux phases (claire et sombre) avec leurs lieux (thylakoïdes, stroma)
- Mentionne ATP, chlorophylle, CO2, glucose, O2
- Explique les mécanismes avec vocabulaire scientifique précis
→ C'est AVANCÉ même s'il manque des détails mineurs

INTERMÉDIAIRE — si l'étudiant :
- Connaît les bases (soleil + CO2 + eau = glucose + O2)
- Cite chloroplastes mais sans détails sur les phases
→ C'est INTERMÉDIAIRE

DÉBUTANT — si l'étudiant :
- Donne une réponse vague sans vocabulaire scientifique
- Dit juste "les plantes utilisent le soleil"
→ C'est DÉBUTANT

Réponds TOUJOURS dans ce format :
NIVEAU : [Débutant / Intermédiaire / Avancé]
CE QUI EST BIEN COMPRIS :
- [point 1]
POINTS ENCORE FLOUS :
- [point 1]
SUGGESTION :
[Une phrase courte]"""

    user_prompt = f"""Question : {question_initiale}
Réponse de l'étudiant : {reponse_etudiant}"""

    return _appeler_llm(system_prompt, user_prompt, temperature=0.1)

print("agent_evaluateur() défini, faire maintenant le test.")

agent_evaluateur() défini, faire maintenant le test.


### Test de `agent_evaluateur()`

In [12]:
question_test = "Explique la photosynthèse"
print(f"Sujet : « {question_test} »")

reponse_etudiant_test = """
La photosynthèse c'est le processus par lequel les plantes utilisent la lumière du soleil, le CO2 et l'eau pour produire du glucose et de l'oxygène. Ça se passe dans les chloroplastes des feuilles.
"""
print(f"\nRéponse de l'étudiant à évaluer : {reponse_etudiant_test}")

# Debutant : La photosynthèse c'est quand les plantes utilisent le soleil pour faire quelque chose avec du CO2. Elles produisent de l'oxygène je crois. Ça se passe dans les feuilles.
# Intermédiaire : La photosynthèse c'est le processus par lequel les plantes utilisent la lumière du soleil, le CO2 et l'eau pour produire du glucose et de l'oxygène. Ça se passe dans les chloroplastes des feuilles.
# Avancé : La photosynthèse est un processus en deux phases : la phase claire dans les thylakoïdes où la chlorophylle absorbe la lumière pour produire de l'ATP et libérer de l'O2 par photolyse de l'eau, et la phase sombre dans le stroma où le CO2 est réduit en glucose via le cycle de Calvin.

evaluation = agent_evaluateur(reponse_etudiant_test, question_test)

print("ÉVALUATION DE LA RÉPONSE")
print("=" * 60)
print(evaluation)

Sujet : « Explique la photosynthèse »

Réponse de l'étudiant à évaluer : 
La photosynthèse c'est le processus par lequel les plantes utilisent la lumière du soleil, le CO2 et l'eau pour produire du glucose et de l'oxygène. Ça se passe dans les chloroplastes des feuilles.

ÉVALUATION DE LA RÉPONSE
NIVEAU : Intermédiaire
CE QUI EST BIEN COMPRIS :
- L'étudiant connaît les bases de la photosynthèse (soleil + CO2 + eau = glucose + O2)
- Il mentionne les chloroplastes comme lieu de la photosynthèse
POINTS ENCORE FLOUS :
- L'étudiant ne détaille pas les phases de la photosynthèse (claire et sombre)
- Il ne mentionne pas les thylakoïdes et le stroma, ni les rôles de la chlorophylle et de l'ATP
SUGGESTION :
Étudiez les phases de la photosynthèse pour approfondir vos connaissances.


## Agent 4 : `generer_indice(question, niveau_indice)`
Comme l'agent 2 ne donne jamais la réponse, létudiant est parfois bloqué (il ne sait vraiment pas la réponse) donc on lui donne un indice.
Agent 2 appelle Agent 4 dans la partie de app.py

**Rôle :** Si l'étudiant ne trouve pas, on lui donne un indice adapté à son niveau de blocage. Plus `niveau_indice` est élevé, plus l'indice est proche de la réponse.

| Niveau | Description |
|---|---|
| `1` | Indice très discret — juste une piste de réflexion |
| `2` | Indice modéré — précise un concept clé |
| `3` | Indice fort — donne presque la réponse |

In [14]:
def generer_indice(question: str, niveau_indice: int) -> str:
    """
    Génère un indice progressif pour aider l'étudiant bloqué.

    Paramètres
    ----------
    question      : str — la question sur laquelle l'étudiant bloque
    niveau_indice : int — niveau de 1 (discret) à 3 (presque la réponse)

    Retourne
    --------
    str — un indice adapté au niveau de blocage
    """
    if niveau_indice not in [1, 2, 3]:
        raise ValueError(f"niveau_indice doit être 1, 2 ou 3. Reçu : {niveau_indice}")

    descriptions_niveau = {
        1: "très discret — pose une question de réflexion générale sans mentionner de concept précis",
        2: "modéré — mentionne un concept clé lié à la réponse, mais ne l'explique pas",
        3: "fort — donne une explication partielle très proche de la réponse complète",
    }

    system_prompt = f"""Tu es un tuteur pédagogique bienveillant.
Tu génères un indice de niveau {niveau_indice}/3 pour aider un étudiant bloqué.

Niveau {niveau_indice} signifie : indice {descriptions_niveau[niveau_indice]}.

Règles :
- L'indice doit tenir en 2-3 phrases maximum.
- Commence par une phrase d'encouragement ("Tu es sur la bonne voie...", "Pense à...", etc.)
- Ne dis jamais explicitement la réponse finale.
- Langue : français."""

    user_prompt = f"""L'étudiant est bloqué sur la question : {question}

Génère un indice de niveau {niveau_indice}/3."""

    return _appeler_llm(system_prompt, user_prompt, temperature=0.6)


print("generer_indice() défini, faire maintenant le test.")

generer_indice() défini, faire maintenant le test.


### Test de `generer_indice()`

In [15]:
question_bloquee = "Quel est le rôle de la chlorophylle dans la photosynthèse ?"

for niveau in [1, 2, 3]:
    indice = generer_indice(question_bloquee, niveau)
    print(f"{'='*60}")
    print(f"INDICE NIVEAU {niveau}/3")
    print(f"{'='*60}")
    print(indice)
    print()

INDICE NIVEAU 1/3
Pense à ce qui se passe lorsque les plantes sont exposées à la lumière du soleil. Qu'est-ce qui leur permet de transformer cette énergie en quelque chose de précieux pour leur croissance ? Réfléchis à la façon dont les plantes utilisent les éléments de leur environnement pour survivre et se développer.

INDICE NIVEAU 2/3
Tu es sur la bonne voie en cherchant à comprendre le processus de photosynthèse ! Pense à l'absorption de la lumière solaire et à la manière dont la chlorophylle intervient dans cette étape cruciale. La chlorophylle joue un rôle clé dans la conversion de l'énergie lumineuse en énergie chimique.

INDICE NIVEAU 3/3
Tu es sur la bonne voie en considérant le processus de photosynthèse ! Pense à la manière dont la chlorophylle absorbe spécifiquement certaines longueurs d'onde de la lumière solaire pour initier la conversion de l'énergie lumineuse en énergie chimique. La chlorophylle joue un rôle clé dans la captation de l'énergie lumineuse, qui est ensuite

---
## Export du module (pour les coéquipiers)
Génèrer `ai_engine.py` depuis ce notebook pour que Membre 3 puisse l'importer dans `app.py`.

In [22]:
import glob
print(glob.glob("*.ipynb"))

['ai_engine (1).ipynb', 'rag_pipeline.ipynb']


In [24]:
import json

with open("ai_engine (1).ipynb", "r", encoding="utf-8") as f:
    notebook = json.load(f)

with open("ai_engine.py", "w", encoding="utf-8") as f:
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            f.write("".join(cell["source"]))
            f.write("\n\n")

print("✅ ai_engine.py créé.")

✅ ai_engine.py créé.
